In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sys.path.append('../src')
from utils import mean_imputation, create_missing_data
from gmm_missing import GMMMissing

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

In [ ]:
data = pd.read_csv('../data/CC GENERAL.csv')
data.drop('CUST_ID', axis=1, inplace=True)
data.head(2)


In [ ]:
#random create missing data 30% missing
X_true, X_incomplete, mask_missing = create_missing_data(data,0.3)
print("Số lượng giá trị thiếu sau khi tạo dữ liệu thiếu:\n", np.isnan(X_incomplete).sum())

In [ ]:
# Nội suy thô bằng Mean
X_initial = mean_imputation(X_incomplete)
scaler = StandardScaler()
X_initial = scaler.fit_transform(X_initial)

gmm = GMMMissing(n_components=3,max_iter=1, tol=1e-4, random_state=42)
gmm._initialize_parameters(X_initial)

In [ ]:
print("Trung bình các cụm (mu) khởi tạo từ K-Means:\n", gmm.mu_)


In [ ]:
log_likelihood, gamma = gmm._e_step(X_initial)
print("Xác suất hậu nghiệm gamma (5 mẫu đầu tiên):\n", gamma[:5])

In [ ]:
gmm._m_step_params(X_initial, gamma)
print("Tâm cụm sau 1 vòng M-Step:\n", gmm.mu_)

In [ ]:
# --- TRỰC QUAN HÓA LOGIC PHƯƠNG TRÌNH 14 TRONG gmm_missing.py ---
# Lấy Sample số 0 làm ví dụ
sample_index = 0

# Tách mask cho sample 0
m_part = mask_missing[sample_index]     # Các chiều bị missing (True)
o_part = ~mask_missing[sample_index]    # Các chiều có dữ liệu (False)

if np.any(m_part):
    print(f"Sample {sample_index} gốc có NaNs: {X_incomplete[sample_index]}")
    x_j_o = X_initial[sample_index, o_part]
    
    # Tổng tích lũy ma trận theo k (k=3 cụm)
    sum_L_mm = np.zeros((np.sum(m_part), np.sum(m_part)))
    sum_term2 = np.zeros(np.sum(m_part))
    
    for k in range(gmm.n_components):
        gamma_jk = gamma[sample_index, k]
        Lambda_k = np.linalg.inv(gmm.covariances_[k])
        
        # Trích xuất L_mm và L_mo (Phương trình 13)
        L_mm = Lambda_k[np.ix_(m_part, m_part)]
        L_mo = Lambda_k[np.ix_(m_part, o_part)]
        
        mu_m = gmm.mu_[k, m_part]
        mu_o = gmm.mu_[k, o_part]
        
        # Phương trình 14 tích lũy
        sum_L_mm += gamma_jk * L_mm
        term2 = L_mm @ mu_m - L_mo @ (x_j_o - mu_o)
        sum_term2 += gamma_jk * term2
        
    # Cập nhật kết quả cuối cùng
    x_j_m_updated = np.linalg.inv(sum_L_mm) @ sum_term2
    
    print(f"Dữ liệu nội suy thô (Mean): {X_initial[sample_index]}")
    print(f"Tọa độ bị thiếu NAY ĐÃ ĐƯỢC KÉO về mốc mới: {x_j_m_updated}")

In [ ]:
# Fit model
gmm_full = GMMMissing(n_components=3, max_iter=200, random_state=42)
gmm_full.fit_scale(X_incomplete)

print(f"Thuật toán hội tụ sau {gmm_full.n_iter_} vòng lặp.")

# Predict cluster
y_pred = gmm_full.predict(gmm_full.X_final_)


plt.figure(figsize=(7, 6))

plt.scatter(
    gmm_full.X_final_[:, 0], 
    gmm_full.X_final_[:, 1], 
    c=y_pred, cmap='viridis', s=30, alpha=0.6
)

# highlight các điểm từng bị missing
missing_rows_idx = np.where(mask_missing.any(axis=1))[0]
plt.scatter(
    gmm_full.X_final_[missing_rows_idx, 0],
    gmm_full.X_final_[missing_rows_idx, 1],
    facecolors='none', edgecolors='red', s=60,
    label='Imputed samples'
)

plt.title("GMM Clustering + Dynamic Imputation")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(gmm_full.X_final_, y_pred)
print("Silhouette score:", score)